# Figure 4

**Applying Mycol to facilitate reproducibility and reporting in mussel larvae**

- **Panels** — **a** workflow strip (seeded geometry) · **b** larvae crops, five per class ·
  **c** DenseNet confusion matrix · **d** manual vs Mycol length · **e** area by class ·
  **f** descriptor t-SNE
- **Needs** — the `case_study_3/` session (**b, c, e, f**); `assets/` for the hand measurements
  (panel **d**)
- **Writes** — `output/Figure_4.svg` and its 300 dpi `.png`, the raster the manuscript embeds;
  intermediates in `output/panels/`
- **Kernel** — `mycol_colonies_env`, run top to bottom


In [ ]:
import re
import subprocess
from pathlib import Path

from PIL import Image

ASSETS = Path("assets")
OUTPUT = Path("output")
PANELS = OUTPUT / "panels"          # intermediates: the workflow strip and the data panels
REPO_ROOT = Path.cwd().parents[2]
PANELS.mkdir(parents=True, exist_ok=True)


def wrote(path):
    print(f"  wrote {path.name:38} {path.stat().st_size:>9,} B")


def rasterise(svg_path, png_path, dpi=300):
    """Render the finished SVG to PNG at `dpi` with the Chrome kaleido installs.

    Chrome is already a dependency - plotly's static export drives it. SVG user units
    are CSS px at 96 dpi, hence the device scale factor of dpi/96.
    """
    try:
        from choreographer.cli import _cli_utils
        chrome = str(_cli_utils.get_chrome_sync())
    except Exception:
        chrome = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
        if not Path(chrome).exists():
            raise RuntimeError("no Chrome - run `plotly_get_chrome`, or install Google Chrome")

    svg = svg_path.read_text()
    w, h = (round(float(v)) for v in
            re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"', svg[:800]).groups())
    page = OUTPUT / "_render.html"
    page.write_text("<!doctype html><meta charset=utf-8><style>"
                    "html,body{margin:0;padding:0;background:#fff}svg{display:block}</style>"
                    + svg[svg.index("<svg"):])
    # Chrome reserves ~87 CSS px of window furniture even headless and clips the page
    # by that much, so render into a taller window and crop back.
    scale, raw = dpi / 96, OUTPUT / "_render.png"
    subprocess.run([chrome, "--headless", "--disable-gpu", "--hide-scrollbars",
                    f"--force-device-scale-factor={scale}", f"--window-size={w},{h + 200}",
                    f"--screenshot={raw.resolve()}", page.resolve().as_uri()],
                   check=True, capture_output=True)
    with Image.open(raw) as im:
        im.crop((0, 0, round(w * scale), round(h * scale))).save(png_path)
    raw.unlink()
    page.unlink()
    wrote(png_path)


## Panel a - the workflow strip

Six step panels plus the flow that composes them, all from seeded geometry, so the strip is
deterministic. The assembly cell inlines `figure4_flow.svg`.


In [ ]:
import math
import random

PREFIX = "figure4"
PANEL_W, PANEL_H = 240, 250
ART_X, ART_Y, ART_S = 30, 48, 180   # the 180x180 art square, in authoring coords
ART_SHIFT = -28                     # art sits high in the panel, title below it
TITLE_Y = 224                       # title baseline
GAP, PAD, STRIP_MARGIN = 52, 28, 16

STRIP_STYLE = """
    text     { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               Helvetica, Arial, sans-serif; fill:#0f172a; }
    .title   { font-size:13px; font-weight:600; text-anchor:middle; }

    .panel   { fill:#ffffff; stroke:#cbd5e1; stroke-width:1.5; }
    .frame   { fill:#ffffff; stroke:#0f172a; stroke-width:1.2; }

    /* larvae: raw, masked, and the two classes */
    .cell    { fill:#94a3b8; fill-opacity:0.30; stroke:#0f172a; stroke-width:0.9; }
    .mask    { fill:#e5431e; fill-opacity:0.17; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .clsA    { fill:#5289C7; fill-opacity:0.45; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .clsB    { fill:#4EB265; fill-opacity:0.45; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .cursor  { fill:#0f172a; stroke:#ffffff; stroke-width:1.1; stroke-linejoin:round; }

    .err     { fill:none; stroke:#c2410c; stroke-width:1.6; stroke-dasharray:4 3.5; }
    .axis    { fill:none; stroke:#94a3b8; stroke-width:1.2; }
    .med     { fill:none; stroke:#0f172a; stroke-width:1.8; stroke-linecap:round; }
    .measure { fill:none; stroke:#0f172a; stroke-width:1.6; }
    .tick    { fill:none; stroke:#0f172a; stroke-width:1.2; stroke-dasharray:3 3; }
    .arrow   { fill:none; stroke:#334155; stroke-width:1.8; }
    .arrowh  { fill:#334155; }
"""

CURSOR = ('<path class="cursor" transform="translate({x},{y}) scale({s})" '
          'd="M 0,0 0,15 3.9,11.4 6.7,17 9.3,15.7 6.5,10.3 11.9,10.1 Z" />')


def harrow(x1, x2, y):
    return (f'<path class="arrow" d="M {x1},{y} H {x2 - 7}" />'
            f'<path class="arrowh" d="M {x2},{y} {x2 - 8},{y - 4.8} {x2 - 8},{y + 4.8} Z" />')


def d_shape(cx, cy, w, h, rot=0):
    """A D: flat edge on the left, true semicircular bulge to the right.

    Two cubics with the circular magic constant, so the curved side reads as a
    half-disc and the flat side stays unmistakably straight.
    """
    x0, y0, y1 = cx - w / 2, cy - h / 2, cy + h / 2
    xm, ym, r, K = cx + w / 2, cy, h / 2, 0.5523
    d = (f"M {x0:.1f},{y0:.1f} "
         f"C {x0 + w * K:.1f},{y0:.1f} {xm:.1f},{ym - r * K:.1f} {xm:.1f},{ym:.1f} "
         f"C {xm:.1f},{ym + r * K:.1f} {x0 + w * K:.1f},{y1:.1f} {x0:.1f},{y1:.1f} Z")
    return d, f'rotate({rot:.1f},{cx:.1f},{cy:.1f})'


def circle_path(cx, cy, r):
    """A circle as a path, so every organism is drawn the same way."""
    return (f'M {cx + r:.1f},{cy:.1f} A {r:.1f},{r:.1f} 0 1 0 {cx - r:.1f},{cy:.1f} '
            f'A {r:.1f},{r:.1f} 0 1 0 {cx + r:.1f},{cy:.1f} Z')


def violin(cx, y0, y1, wa, wb, med, cls):
    """Blunt-ended violin with a median bar - matches the pipeline figure."""
    h = y1 - y0
    up, lo = y0 + h * 0.18, y0 + h * 0.62
    return (f'<path class="{cls}" d="M {cx - 3},{y0} '
            f'C {cx - wa},{up} {cx - wb},{lo} {cx - 2.5},{y1} '
            f'L {cx + 2.5},{y1} '
            f'C {cx + wb},{lo} {cx + wa},{up} {cx + 3},{y0} Z" />'
            f'<path class="med" d="M {cx - 8},{med} H {cx + 8}" />')


In [ ]:
# ── the larvae field: eight larvae, two of them abnormal, spaced so none touches
#    the frame even when one mask is drawn 34% too large ──
BAD_MASK = 3                                     # this outline needs fixing
_LV_H = (26, 31)
_REACH = math.hypot(_LV_H[1] * 0.62, _LV_H[1]) / 2 * 1.34 + 2

_rng = random.Random(5)
_pts, _guard = [], 0
while len(_pts) < 8 and _guard < 8000:           # rejection sampling, min 44 apart
    _guard += 1
    p = (_rng.uniform(ART_X + _REACH, ART_X + ART_S - _REACH),
         _rng.uniform(ART_Y + _REACH, ART_Y + ART_S - _REACH))
    if all(math.dist(p, q) >= 44 for q in _pts):
        _pts.append(p)

LARVAE = []
for i, (x, y) in enumerate(_pts):
    h = _rng.uniform(*_LV_H)
    LARVAE.append({"x": x, "y": y, "normal": i % 4 != 2,      # most are normal
                   "w": h * 0.62, "h": h, "rot": _rng.uniform(0, 360)})


def larva_svg(lv, cls, inflate=1.0):
    """Normal larvae are D-shaped; abnormal ones are round."""
    if lv["normal"]:
        d, tr = d_shape(lv["x"], lv["y"], lv["w"] * inflate, lv["h"] * inflate, lv["rot"])
        return f'<path class="{cls}" d="{d}" transform="{tr}" />'
    return f'<path class="{cls}" d="{circle_path(lv["x"], lv["y"], lv["h"] * 0.42 * inflate)}" />'


def larvae_field(cls_for, inflate_for=None):
    out = [f'<rect class="frame" x="{ART_X}" y="{ART_Y}" width="{ART_S}" height="{ART_S}" />']
    out += [larva_svg(lv, cls_for(i, lv), inflate_for(i) if inflate_for else 1.0)
            for i, lv in enumerate(LARVAE)]
    return "\n  ".join(out)


def _loose(i):
    return 1.34 if i == BAD_MASK else 1.0


def step3_classify():
    """DenseNet calls: normal blue, abnormal green - one call is wrong.

    The flagged larva is D-shaped, so it should be blue; the classifier puts it in the
    abnormal class instead and it comes out green.
    """
    return larvae_field(lambda i, lv: "clsA" if (not lv["normal"] if i == BAD_MASK
                                                 else lv["normal"]) else "clsB", _loose)


def step4_curate():
    """The loose outline tightened and the class put right: green back to blue."""
    lv = LARVAE[BAD_MASK]
    return (larvae_field(lambda i, l: "clsA" if l["normal"] else "clsB") + "\n  "
            + f'<circle class="err" cx="{lv["x"]:.1f}" cy="{lv["y"]:.1f}" r="26" />'
            + "\n  " + CURSOR.format(x=lv["x"] + 16, y=lv["y"] + 9, s=1.0))


def step5_measure():
    """The longest internal line, parallel to the straight edge of the D."""
    cx, cy, w, h = 118, 138, 66, 106
    d, _ = d_shape(cx, cy, w, h)
    x0, y0, y1 = cx - w / 2, cy - h / 2, cy + h / 2
    xm = x0 + 9                                   # the measured chord, just inside
    return "\n  ".join([
        f'<path class="clsA" d="{d}" />',
        f'<path class="tick" d="M {x0 - 16},{y0} H {xm + 26}" />',
        f'<path class="tick" d="M {x0 - 16},{y1} H {xm + 26}" />',
        f'<path class="measure" d="M {xm},{y0 + 8} V {y1 - 8}" />',
        f'<path class="arrowh" d="M {xm},{y0} {xm - 4.6},{y0 + 9} {xm + 4.6},{y0 + 9} Z" />',
        f'<path class="arrowh" d="M {xm},{y1} {xm - 4.6},{y1 - 9} {xm + 4.6},{y1 - 9} Z" />',
    ])


def step6_distribution():
    """Distribution of that measurement, one violin per class."""
    return "\n  ".join([
        '<path class="axis" d="M 52,66 V 206 H 200" />',
        # abnormal larvae sit lower and span a shorter range than normal ones
        violin(98, 72, 180, 33, 16, 114, "clsA"),
        violin(162, 132, 202, 17, 27, 172, "clsB"),
    ])


STEPS = [("step1_larvae", lambda: larvae_field(lambda i, lv: "cell"), "Mussel larvae images"),
         ("step2_segment", lambda: larvae_field(lambda i, lv: "mask", _loose),
          "Cellpose segmentation"),
         ("step3_classify", step3_classify, "DenseNet classification"),
         ("step4_curate", step4_curate, "Manual curation"),
         ("step5_measure", step5_measure, "Maximum internal length"),
         ("step6_distribution", step6_distribution, "Class distributions")]


In [ ]:
STEP_SVG = """<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}" version="1.1"
     xmlns="http://www.w3.org/2000/svg">
  <title>{title}</title>
  <style>{style}  </style>
  <rect class="panel" x="0.75" y="0.75" width="{iw}" height="{ih}" rx="10" />
  <g transform="translate(0,{shift})">
  {art}
  </g>
  <text class="title" x="{tx}" y="{ty}">{title}</text>
</svg>
"""

# one standalone SVG per step, and the same bodies composed into the flow
bodies = []
for name, fn, title in STEPS:
    svg = STEP_SVG.format(w=PANEL_W, h=PANEL_H, iw=PANEL_W - 1.5, ih=PANEL_H - 1.5,
                          style=STRIP_STYLE, art=fn(), title=title,
                          tx=PANEL_W / 2, ty=TITLE_Y, shift=ART_SHIFT)
    (PANELS / f"{PREFIX}_{name}.svg").write_text(svg)
    inner = re.sub(r"^.*?<svg[^>]*>", "", svg, flags=re.S).rsplit("</svg>", 1)[0]
    bodies.append(re.sub(r"<style>.*?</style>", "", inner, flags=re.S).strip())

n = len(STEPS)
band_w = PAD * 2 + n * PANEL_W + (n - 1) * GAP
W = STRIP_MARGIN * 2 + band_w
H = STRIP_MARGIN * 2 + PAD * 2 + PANEL_H
y = STRIP_MARGIN + PAD
xs = [STRIP_MARGIN + PAD + i * (PANEL_W + GAP) for i in range(n)]

parts = [f'<rect x="0" y="0" width="{W}" height="{H}" fill="#ffffff" />',
         f'<rect x="{STRIP_MARGIN}" y="{STRIP_MARGIN}" width="{band_w}" '
         f'height="{PAD * 2 + PANEL_H}" rx="18" fill="#f5f8ff" '
         f'stroke="#dbe6fb" stroke-width="1" />']
parts += [harrow(xs[i] + PANEL_W + 10, xs[i + 1] - 10, y + PANEL_H / 2) for i in range(n - 1)]
parts += [f'<g transform="translate({x},{y})">\n{body}\n  </g>'
          for x, body in zip(xs, bodies)]

(PANELS / f"{PREFIX}_flow.svg").write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{W}" height="{H}" viewBox="0 0 {W} {H}" version="1.1"\n'
    f'     xmlns="http://www.w3.org/2000/svg">\n'
    f'  <style>{STRIP_STYLE}  </style>\n' + "\n  ".join(parts) + "\n</svg>\n")

print(f"  {n} step SVGs + {PREFIX}_flow.svg ({W}x{H})")
wrote(PANELS / f"{PREFIX}_flow.svg")


## Panels b to f - the data

Agg is forced - otherwise matplotlib picks the macOS backend, applies 2x Retina scaling and snaps
figure sizes to whole device pixels, shifting a panel by a couple of pixels.


In [ ]:
import io
import json
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SESSION_ZIP = REPO_ROOT / "case_study_3" / "mycol_saved_session_CS3.zip"
MERGED_CSV = ASSETS / "manual_vs_mycol_larvae_measurements.csv"   # manual ImageJ lengths
for p in (SESSION_ZIP, MERGED_CSV):
    shown = p.relative_to(REPO_ROOT) if p.is_absolute() else p
    print(f"{'ok     ' if p.exists() else 'MISSING'}  {shown}")

session = zipfile.ZipFile(SESSION_ZIP)
metrics = pd.read_csv(io.BytesIO(session.read("cell_metrics.csv")))
labelled = metrics[metrics["mask label"] != "Unlabelled"].reset_index(drop=True)

print(f"\n{len(metrics)} cells over {metrics['image'].nunique()} images; "
      f"{len(labelled)} labelled, {len(metrics) - len(labelled)} not")
print("  " + ", ".join(f"{k} {v}" for k, v in labelled["mask label"].value_counts().items()))


In [ ]:
# --- the layout the assembly cell below uses, and the type size it implies ---
FIG_W, MARGIN, GAP_X = 1788, 20, 26
COL_W = (FIG_W - 2 * MARGIN - 2 * GAP_X) / 3        # 565.33 px - one column
WIDE_W = 2 * COL_W + GAP_X                          # 1156.67 px - panel b spans two
TEXT_PX = 19.5                                      # every label in the figure, in px

PANEL_IN = 4.5
WIDE_IN = PANEL_IN * WIDE_W / COL_W
DPI = 300
FONT_PT = TEXT_PX / ((DPI / 72) * (COL_W / (PANEL_IN * DPI)))

# Scale of the captures. Panels d and e report micrometres: mycol's cell_metrics are in
# pixels, and this converts them. It is also the factor already baked into
# manual_vs_mycol_larvae_measurements.csv - the panel d cell checks that, rather than trusting it.
UM_PER_PX = 0.69

# normal is clsA, abnormal is clsB in the workflow strip above
CLASS_COLOURS = {"normal": "#5289C7", "abnormal": "#4EB265"}
CLASS_ORDER = ["normal", "abnormal"]

plt.rcParams.update({"font.size": FONT_PT, "axes.labelsize": FONT_PT,
                     "xtick.labelsize": FONT_PT, "ytick.labelsize": FONT_PT,
                     "legend.fontsize": FONT_PT})


def save_panel(fig, stem):
    """Write a panel as PNG (what the figure embeds) and SVG (for editing)."""
    for ext in ("png", "svg"):
        fig.savefig(PANELS / f"{stem}.{ext}", dpi=DPI, facecolor="white")
    px = Image.open(PANELS / f"{stem}.png").size
    print(f"  wrote {stem}.png / .svg  ({px[0]}x{px[1]} px)")


print(f"square panels {PANEL_IN}x{PANEL_IN} in, wide panel {WIDE_IN:.3f}x{PANEL_IN} in, {DPI} dpi")
print(f"FONT_PT = {FONT_PT:.2f}  ->  {TEXT_PX} px in the finished figure")
print(f"pixel scale {UM_PER_PX} um/px  ->  areas x {UM_PER_PX ** 2:.4f} for um\u00b2")


In [ ]:
# --- panels d, e and f share one plot rectangle, so they read as a row ---
# All three are the same square in the finished figure, but that only reads as a row if
# the plot areas inside those squares line up too, and left to itself each comes out
# different: constrained layout sizes the axes around whatever tick labels that panel
# carries, and panel f's equal aspect shrinks its box again to match the embedding. The
# rectangle they can share is the largest that fits all three, so it is not known until
# every one has been laid out - hence holding them back.
BOTTOM_ROW = []

# The rectangle this notebook last wrote. Passing it to save_bottom_row() pins the row
# where it already sits: the measurement depends on the renderer, so re-running these
# cells outside the inline backend lands ~0.006 lower and moves all three panels.
# Pass rect=None instead to measure it afresh.
COMMITTED_RECT = (0.18927478122770716, 0.11525507381386783,
                  0.8014652187722928, 0.8014652187722928)


def hold_panel(fig, ax, stem):
    fig.canvas.draw()                       # constrained layout only settles on a draw
    BOTTOM_ROW.append((fig, ax, stem))


def save_bottom_row(rect=None):
    """Put every held panel on one plot rectangle, then write them out."""
    if rect is None:
        boxes = [ax.get_position() for _, ax, _ in BOTTOM_ROW]
        x0, x1 = max(b.x0 for b in boxes), min(b.x1 for b in boxes)
        y0, y1 = max(b.y0 for b in boxes), min(b.y1 for b in boxes)
        side = min(x1 - x0, y1 - y0)        # squared off: panels d and f are equal-aspect
        rect = [x0, y0, side, side]

    for fig, ax, stem in BOTTOM_ROW:
        fig.set_layout_engine("none")       # else the engine re-runs on save and undoes this
        ax.set_position(rect)
        save_panel(fig, stem)
    return list(rect)


In [ ]:
# --- panel b: five normal larvae across the size range, five unmistakably abnormal ones ---
N_PER_CLASS = 5
PERCENTILES = np.linspace(10, 90, N_PER_CLASS)       # 10, 30, 50, 70, 90  (normal class)
PATCH_PX = 256                                       # display size of one crop
PATCH_MARGIN = 0.35                                  # extra field around the mask's box
GUTTER = 12
ABNORMAL_POOL = 30                                   # candidates before focus and plate filters

# scored without any size descriptor: what separates a D-larva from a failed one is its
# outline, and size is panel e's subject
SHAPE_ONLY = ["circularity", "roundness", "solidity", "extent", "eccentricity"]

_image_cache = {}
_members = set(session.namelist())


def member(folder, stem):
    """Find one image/mask inside the session, whichever way it names its files.

    Sessions are not consistent: some store masks/<stem>.tif, later ones
    masks/<stem>_masks.tif. Resolving it here means a re-export does not break the panel.
    """
    for candidate in (f"{folder}/{stem}.tif", f"{folder}/{stem}_masks.tif"):
        if candidate in _members:
            return candidate
    raise KeyError(f"no {folder} member for {stem} in {SESSION_ZIP.name}")


def patch_for(row):
    """A square crop of the image around one cell's mask box - background left alone.

    The mask says where to cut and nothing more; nothing is blanked out. The box is
    squared off on its longer side and grown by PATCH_MARGIN, then clipped to the frame,
    so the larva sits centred in a little of its own field.
    """
    stem = row["image"].removesuffix(".tif")
    if stem not in _image_cache:
        _image_cache[stem] = (
            Image.open(io.BytesIO(session.read(member("images", stem)))).convert("RGB"),
            np.array(Image.open(io.BytesIO(session.read(member("masks", stem))))),
        )
    img, mask = _image_cache[stem]

    ys, xs = np.where(mask == int(row["mask #"]))
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    half = max(ys.max() - ys.min(), xs.max() - xs.min()) / 2 * (1 + PATCH_MARGIN)

    # keep the crop square even at the frame edge: shift the centre, don't shrink the box
    H, W = mask.shape
    cx, cy = min(max(cx, half), W - half), min(max(cy, half), H - half)
    box = (round(cx - half), round(cy - half), round(cx + half), round(cy + half))
    return np.array(img.crop(box).resize((PATCH_PX, PATCH_PX), Image.LANCZOS))


def focus_of(patch):
    """Variance of the Laplacian - low means the larva sat outside the focal plane."""
    g = np.asarray(Image.fromarray(patch).convert("L"), float)
    lap = g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:] - 4 * g[1:-1, 1:-1]
    return float(lap.var())


def by_area_percentile(sub):
    """Nearest cell to each target area, without picking the same one twice."""
    picks, taken = [], set()
    for t in np.percentile(sub["area"], PERCENTILES):
        pick = next(i for i in (sub["area"] - t).abs().sort_values().index if i not in taken)
        taken.add(pick)
        picks.append(sub.loc[pick])
    return picks


def least_d_shaped(sub, everything):
    """The abnormal larvae that most visibly are abnormal - round, in focus, one per culture."""
    X = everything[SHAPE_ONLY].to_numpy(float)
    Z = (X - X.mean(0)) / X.std(0)
    is_ab = (everything["mask label"] == "abnormal").to_numpy()
    w = Z[is_ab].mean(0) - Z[~is_ab].mean(0)          # abnormal - normal, in z units
    w /= np.linalg.norm(w)
    score = pd.Series(Z @ w, index=everything.index)

    pool = sub.assign(shape_score=score.loc[sub.index]).nlargest(ABNORMAL_POOL, "shape_score")
    pool = pool.assign(focus=[focus_of(patch_for(r)) for _, r in pool.iterrows()])
    sharp = pool[pool["focus"] >= pool["focus"].median()]
    # experiment2_C1_Snap-2854.tif -> experiment2_C1, the culture the larva came from
    best = (sharp.assign(culture=sharp["image"].map(
                lambda n: "_".join(n.removesuffix(".tif").split("_")[:2])))
                 .sort_values("shape_score", ascending=False)
                 .drop_duplicates(subset="culture", keep="first"))
    return [best.loc[i] for i in best.head(N_PER_CLASS).index]


In [ ]:
chosen = {"normal": by_area_percentile(labelled[labelled["mask label"] == "normal"]),
          "abnormal": least_d_shaped(labelled[labelled["mask label"] == "abnormal"], labelled)}
for cls in CLASS_ORDER:
    print(f"  {cls}:")
    for p in chosen[cls]:
        extra = (f"  shape {p['shape_score']:5.2f}  circularity {p['circularity']:.3f}"
                 f"  eccentricity {p['eccentricity']:.3f}") if "shape_score" in p else ""
        print(f"    {p['image']:32s} m{int(p['mask #'])}  area {int(p['area']):7,}{extra}")

# montage: one row per class, five patches across, white gutters
rows = []
for cls in CLASS_ORDER:
    gap = np.full((PATCH_PX, GUTTER, 3), 255, np.uint8)
    imgs = [patch_for(p) for p in chosen[cls]]
    rows.append(np.hstack([x for im in imgs for x in (im, gap)][:-1]))
montage = np.vstack([rows[0], np.full((GUTTER, rows[0].shape[1], 3), 255, np.uint8), rows[1]])

fig, ax = plt.subplots(figsize=(WIDE_IN, PANEL_IN), layout="constrained")
# plt.subplots snaps the size when this runs as a script rather than under the inline
# backend; setting it again keeps the panel at the width the other wide panels have
fig.set_size_inches(WIDE_IN, PANEL_IN)
ax.imshow(montage)
ax.set_yticks([PATCH_PX / 2, PATCH_PX * 1.5 + GUTTER], [c.capitalize() for c in CLASS_ORDER])
ax.set_xticks([])
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
# colour each row label to match its class everywhere else in the figure
for tick, cls in zip(ax.get_yticklabels(), CLASS_ORDER):
    tick.set_color(CLASS_COLOURS[cls])
    tick.set_fontweight("bold")
save_panel(fig, "larvae_patches")


In [ ]:
# --- panel c: the DenseNet confusion matrix the app stored with the session ---
assert "densenet_confusion_matrix.json" in _members, (
    "the session stores no confusion matrix - re-train the DenseNet in mycol and re-export it")

trace = json.loads(session.read("densenet_confusion_matrix.json"))["data"][0]
cmat = np.array([[int(v) for v in row] for row in trace["text"]])   # rows true, cols predicted
names = list(trace["x"])
share = cmat / cmat.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
for i in range(cmat.shape[0]):
    for j in range(cmat.shape[1]):
        ax.text(j, i, f"{cmat[i, j]:,}", ha="center", va="center", fontsize=FONT_PT,
                color="white" if share[i, j] > 0.5 else "#0f172a")
ax.set_xticks(range(len(names)), names)
ax.set_yticks(range(len(names)), names, rotation=90, va="center")
ax.set_xlabel("Predicted Class", fontsize=FONT_PT)
ax.set_ylabel("True Class", fontsize=FONT_PT)
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.text(0, 1.04, f"Accuracy {np.trace(cmat) / cmat.sum() * 100:.1f}%   n = {cmat.sum():,}",
        transform=ax.transAxes, ha="left", va="bottom", fontsize=FONT_PT)
save_panel(fig, "larvae_confusion_matrix")

print(f"  {cmat.sum():,} held-out patches, "
      f"accuracy {np.trace(cmat) / cmat.sum() * 100:.1f}%")
for i, c in enumerate(names):
    print(f"    {c:9s} recall {share[i, i] * 100:5.1f}%  ({cmat[i, i]:,} of {cmat[i].sum():,})")


In [ ]:
# --- panel d: manual ImageJ length vs mycol's major axis, same larvae, in micrometres ---
merged = pd.read_csv(MERGED_CSV)
X_MANUAL, Y_MYCOL = "Manual Length", "Mycol: Major Axis Length"

# Check the CSV really is on the um scale this notebook assumes: the session's pixel value
# over this column must come out at 1 / UM_PER_PX, or the axis labels below would be wrong.
scale = merged.merge(metrics[metrics["mask #"] == 1][["image", "major axis length"]], on="image")
factor = (scale["major axis length"] / scale[Y_MYCOL]).mean()
assert abs(1 / factor - UM_PER_PX) < 1e-4, (
    f"the CSV is {1 / factor:.4f} um/px but UM_PER_PX is {UM_PER_PX} - a different magnification")
print(f"  scale check: session px / csv = {factor:.6f}  ->  {1 / factor:.4f} um/px "
      f"(n={len(scale)}) - both columns are already um")

r = merged[X_MANUAL].corr(merged[Y_MYCOL])
lim = max(merged[X_MANUAL].max(), merged[Y_MYCOL].max()) * 1.05
lo = min(merged[X_MANUAL].min(), merged[Y_MYCOL].min()) * 0.95

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
ax.plot([lo, lim], [lo, lim], color="#8a8a8a", lw=1.2, ls="--", zorder=1)
for cls in CLASS_ORDER:
    d = merged[merged["mask label"] == cls]
    ax.scatter(d[X_MANUAL], d[Y_MYCOL], s=46, alpha=0.8, color=CLASS_COLOURS[cls],
               edgecolors="white", linewidths=0.5, zorder=2,
               label=f"{cls.capitalize()} (n={len(d)})")

ax.set_xlim(lo, lim)
ax.set_ylim(lo, lim)
ax.set_aspect("equal")
ax.text(0.05, 0.95, f"R\u00b2 = {r ** 2:.3f}\nn = {len(merged)}", transform=ax.transAxes,
        va="top", ha="left", fontsize=FONT_PT,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
ax.set_xlabel("Manual Length (\u00b5m)", fontsize=FONT_PT)
ax.set_ylabel("Mycol Major Axis (\u00b5m)", fontsize=FONT_PT)
ax.legend(loc="lower right", frameon=True, framealpha=0.9, edgecolor="#dddddd",
          fontsize=FONT_PT, handletextpad=0.4, borderpad=0.4)
ax.grid(True, lw=0.4, color="#ededed")
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
hold_panel(fig, ax, "manual_vs_mycol")      # written once d, e and f can share a rectangle

bias = (merged[Y_MYCOL] - merged[X_MANUAL]).mean()
print(f"  R\u00b2 = {r ** 2:.3f} over {len(merged)} larvae; mycol reads {bias:+.2f} \u00b5m "
      f"({bias / merged[X_MANUAL].mean() * 100:+.2f}%) vs the hand measure")


In [ ]:
# --- panel e: area distribution per class, in um\u00b2, drawn the way the app draws it ---
VIOLIN_W = 0.75
JITTER = 0.25                       # app's jitter, as a fraction of the violin width
AREA_SCALE = UM_PER_PX ** 2         # px\u00b2 -> um\u00b2; cell_metrics area is in pixels
rng = np.random.default_rng(42)     # seeded so the point cloud is reproducible

data = [labelled.loc[labelled["mask label"] == c, "area"].to_numpy() * AREA_SCALE
        for c in CLASS_ORDER]

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
# violin body only - no box, no mean line, no extrema; the overlaid points carry the detail
parts = ax.violinplot(data, positions=range(len(CLASS_ORDER)), widths=VIOLIN_W,
                      showmeans=False, showmedians=False, showextrema=False)
for body, cls in zip(parts["bodies"], CLASS_ORDER):
    body.set_facecolor(CLASS_COLOURS[cls])
    body.set_alpha(0.85)
    body.set_edgecolor("#0f172a")
    body.set_linewidth(1.2)

# every cell overlaid as a jittered black point, centred on the violin (pointpos=0)
for i, d in enumerate(data):
    x = i + rng.uniform(-JITTER * VIOLIN_W / 2, JITTER * VIOLIN_W / 2, len(d))
    ax.scatter(x, d, s=5, color="black", alpha=0.45, linewidths=0, zorder=3)

ax.set_xticks(range(len(CLASS_ORDER)),
              [f"{c.capitalize()}\n(n={len(d):,})" for c, d in zip(CLASS_ORDER, data)])
ax.set_ylabel("Area (\u00b5m\u00b2)", fontsize=FONT_PT)
ax.grid(True, axis="y", lw=0.4, color="#ededed")
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
hold_panel(fig, ax, "area_violin")          # written once d, e and f can share a rectangle

for c, d in zip(CLASS_ORDER, data):
    print(f"  {c:9s} n={len(d):3d}  median {np.median(d):8,.0f} \u00b5m\u00b2  IQR "
          f"{np.percentile(d, 25):,.0f}-{np.percentile(d, 75):,.0f}")


In [ ]:
# --- panel f: t-SNE over the morphology descriptors ---
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Everything that is not an identifier is a descriptor. Taking them this way rather than
# naming them means a change to what the app exports shows up as a different descriptor
# list, not a KeyError or a silently stale panel.
ID_COLS = {"image #", "image", "mask #", "mask label"}
DESCRIPTORS = [c for c in labelled.columns if c not in ID_COLS]
print("  descriptors:", ", ".join(DESCRIPTORS))

Z = StandardScaler().fit_transform(labelled[DESCRIPTORS])
coords = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42).fit_transform(Z)

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
for cls in CLASS_ORDER:
    m = (labelled["mask label"] == cls).to_numpy()
    ax.scatter(coords[m, 0], coords[m, 1], s=26, alpha=0.75, color=CLASS_COLOURS[cls],
               edgecolors="white", linewidths=0.4, label=f"{cls.capitalize()} (n={int(m.sum())})")

ax.set_xlabel("t-SNE 1", fontsize=FONT_PT)
ax.set_ylabel("t-SNE 2", fontsize=FONT_PT)
ax.set_xticks([])
ax.set_yticks([])
# The embedding is wider than it is tall. adjustable="datalim" keeps one unit of t-SNE 1
# the same length as one unit of t-SNE 2 - so the clusters keep their shape - by padding
# the short axis, rather than the default, which keeps the aspect by shrinking the axes
# box and would leave this panel visibly shorter than d and e beside it.
ax.set_aspect("equal", adjustable="datalim")
ax.legend(loc="upper right", frameon=True, framealpha=0.9, edgecolor="#dddddd",
          fontsize=FONT_PT, handletextpad=0.4, borderpad=0.4)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
hold_panel(fig, ax, "descriptor_tsne")      # written once d, e and f can share a rectangle

print(f"  {len(labelled)} cells over {len(DESCRIPTORS)} descriptors")


In [ ]:
# --- panels d, e and f: one plot rectangle, then written out ---
PLOT_RECT = save_bottom_row(rect=COMMITTED_RECT)
x0, y0, w, h = PLOT_RECT
print(f"\nshared plot area  x0={x0:.4f} y0={y0:.4f} {w:.4f}x{h:.4f} of the panel "
      f"-> {w * COL_W:.0f}x{h * COL_W:.0f} px in the finished figure")


## Assembly

Lays panel **a** and the panels out on the 1788 px canvas, draws the panel letters (here and nowhere
else), and embeds the rasters as data URIs so the `.svg` stands alone. The `.svg` is the master; the
`.png` the manuscript embeds is rendered from it at 300 dpi.


In [ ]:
import base64

OUT = OUTPUT / "Figure_4.svg"

ROW_GAP = 26
LETTER_H, LETTER_SIZE = 36, 30      # strip above each row for its panel letter
CONTENT_W = FIG_W - 2 * MARGIN

FLOW_SVG = PANELS / f"{PREFIX}_flow.svg"
FLOW_MARGIN = 16            # the flow's own gutter, backed out so its band spans the figure

ROW2 = [{"letter": "b", "name": "example larvae", "w": WIDE_W,
         "src": PANELS / "larvae_patches.png"},
        {"letter": "c", "name": "confusion matrix", "w": COL_W,
         "src": PANELS / "larvae_confusion_matrix.png"}]
ROW3 = [{"letter": "d", "name": "length agreement", "w": COL_W,
         "src": PANELS / "manual_vs_mycol.png"},
        {"letter": "e", "name": "area by class", "w": COL_W,
         "src": PANELS / "area_violin.png"},
        {"letter": "f", "name": "descriptor t-SNE", "w": COL_W,
         "src": PANELS / "descriptor_tsne.png"}]

STYLE = """
    text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
           Helvetica, Arial, sans-serif; fill:#0f172a; }
    .fig-letter { font-size:%dpx; font-weight:700; }
""" % LETTER_SIZE


def data_uri(path):
    return f"data:image/png;base64,{base64.b64encode(path.read_bytes()).decode()}"


def inline_flow(x, y, scale):
    """Drop the workflow SVG in as vector, keeping its own <style>.

    `x`/`y` place the flow's *band*, not its canvas: FLOW_MARGIN is backed out so the
    band lines up edge to edge with the panels below. The flow's 13 px titles would land
    at 13*scale px, so they are rewritten to hit TEXT_PX exactly; only this inlined copy
    is touched, and the standalone flow SVG keeps its own sizing.
    """
    inner = re.sub(r"^.*?<svg[^>]*>", "", FLOW_SVG.read_text(), flags=re.S).rsplit("</svg>", 1)[0]
    # drop the flow's white backdrop: pulled flush to the figure edge it would reach up
    # over the panel letter and hide it
    inner = re.sub(r'<rect x="0" y="0" width="\d+" height="\d+" fill="#ffffff" ?/>',
                   "", inner, count=1)
    inner = re.sub(r"(\.title\s*\{[^}]*?font-size:)[\d.]+px",
                   lambda m: f"{m.group(1)}{TEXT_PX / scale:.2f}px", inner, count=1)
    ox, oy = x - FLOW_MARGIN * scale, y - FLOW_MARGIN * scale
    return f'<g transform="translate({ox:.2f},{oy:.2f}) scale({scale:.5f})">{inner}</g>'


print(f"figure width {FIG_W}, column {COL_W:.0f} px, wide panel {WIDE_W:.0f} px")
for spec in ROW2 + ROW3:
    im = Image.open(spec["src"])
    print(f"  panel {spec['letter']}  {spec['name']:<18} {spec['src'].name:<28} "
          f"{im.width}x{im.height} -> {spec['w']:.0f} px wide")

flow_w, flow_canvas_h = (float(v) for v in
                         re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"',
                                   FLOW_SVG.read_text()[:600]).groups())
flow_scale = CONTENT_W / (flow_w - 2 * FLOW_MARGIN)
flow_h = (flow_canvas_h - 2 * FLOW_MARGIN) * flow_scale
print(f"  panel a  {'workflow':<18} {FLOW_SVG.name:<28} {flow_w:.0f}x{flow_canvas_h:.0f} -> "
      f"band {CONTENT_W:.0f} px ({flow_scale:.2f}x)")

# every panel is drawn at the aspect that makes it exactly one column tall, so both
# lower rows are COL_W high and the grid stays square
row_h = COL_W
y_a = MARGIN + LETTER_H
y_b = y_a + flow_h + ROW_GAP + LETTER_H
y_d = y_b + row_h + ROW_GAP + LETTER_H
fig_h = int(round(y_d + row_h + MARGIN))

parts = [f'<rect x="0" y="0" width="{FIG_W}" height="{fig_h}" fill="#ffffff" />',
         f'<text class="fig-letter" x="{MARGIN}" y="{MARGIN + LETTER_SIZE}">a</text>',
         inline_flow(MARGIN, y_a, flow_scale)]

for row, y in ((ROW2, y_b), (ROW3, y_d)):
    x = MARGIN
    for spec in row:
        parts.append(f'<text class="fig-letter" x="{x:.1f}" '
                     f'y="{y - LETTER_H + LETTER_SIZE:.1f}">{spec["letter"]}</text>')
        parts.append(f'<image x="{x:.1f}" y="{y:.1f}" width="{spec["w"]:.2f}" '
                     f'height="{row_h:.2f}" href="{data_uri(spec["src"])}" />')
        x += spec["w"] + GAP_X

OUT.write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{FIG_W}" height="{fig_h}" viewBox="0 0 {FIG_W} {fig_h}"\n'
    f'     version="1.1" xmlns="http://www.w3.org/2000/svg"\n'
    f'     xmlns:xlink="http://www.w3.org/1999/xlink">\n'
    f'  <title>Figure 4 - larvae morphology</title>\n'
    f'  <style>{STYLE}  </style>\n  ' + "\n  ".join(parts) + "\n</svg>\n")

print()
wrote(OUT)
rasterise(OUT, OUTPUT / "Figure_4.png")


## What this notebook wrote


In [ ]:
from IPython.display import display

for p in sorted(OUTPUT.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(OUTPUT)):44} {p.stat().st_size:>9,} B")

im = Image.open(OUTPUT / "Figure_4.png")
print(f"\nFigure 4   {im.width} x {im.height} px")
im.thumbnail((760, 760))
display(im.convert("RGB"))
